In [1]:
import tensorflow as tf

In [2]:
import keras

In [3]:
from keras._tf_keras.keras.preprocessing.image import ImageDataGenerator

In [4]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [21]:
from google.colab import drive
drive.mount('/content/drive')


!unzip '/content/drive/MyDrive/CornSeeds/archive.zip' -d '/content/drive/MyDrive/CornSeeds/corn_data/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/MyDrive/CornSeeds/archive.zip
replace /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_204120329_Bihilifa48.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_204120329_Bihilifa48.jpg  
  inflating: /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_206872470_Bihilifa60.jpg  
  inflating: /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_207756231_Bihilifa53.jpg  
  inflating: /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_214991971_Bihilifa27.jpg  
  inflating: /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_220863168_Bihilifa17.jpg  
  inflating: /content/drive/MyDrive/CornSeeds/corn_data/MaizeData/Bhihilifa/aug_3_224302478_Bihilifa52.jpg  
  inflating: /conte

In [22]:
import os
import shutil


base_path = '/content/drive/MyDrive/CornSeeds/corn_data/MaizeData'
train_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data'

class_mapping = {
    'WangDataa': 'healthy',
    'SanzalSima': 'unhealthy',
    'Bhihilifa': 'unhealthy'
}


for label in set(class_mapping.values()):
    os.makedirs(os.path.join(train_path, label), exist_ok=True)


for source_folder, label in class_mapping.items():
    src = os.path.join(base_path, source_folder)
    dst = os.path.join(train_path, label)

    if os.path.exists(src):
        files = os.listdir(src)
        for f in files:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        print(f"Moved {len(files)} images from {source_folder} to {label}")

print("\nRelabeling complete. New source for training: ", train_path)

Moved 1000 images from WangDataa to healthy
Moved 500 images from SanzalSima to unhealthy
Moved 500 images from Bhihilifa to unhealthy

Relabeling complete. New source for training:  /content/drive/MyDrive/CornSeeds/corn_data/labeled_data


In [23]:
import os
print(os.listdir('/content/drive/MyDrive/CornSeeds/corn_data/labeled_data'))

['unhealthy', 'healthy']


In [24]:
training_set1 = datagen.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.


In [25]:
training_datagenerator1 = ImageDataGenerator(rescale = 1./255)

In [26]:
testing_set1 = training_datagenerator1.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.


In [27]:
cnn_process1 = tf.keras.models.Sequential()

# FIRST LAYER

In [28]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=[64, 64, 3]
    )
)

cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

# SECOND LAYER

In [29]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
    )
)
cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

In [30]:
cnn_process1.add(tf.keras.layers.Flatten())

In [31]:
cnn_process1.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn_process1.add(tf.keras.layers.Dropout(0.5))

In [32]:
cnn_process1.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [33]:
cnn_process1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [34]:
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=100)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 336ms/step - accuracy: 0.5535 - loss: 0.6475 - val_accuracy: 0.6845 - val_loss: 0.5493
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 265ms/step - accuracy: 0.7175 - loss: 0.5380 - val_accuracy: 0.7335 - val_loss: 0.5181
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 316ms/step - accuracy: 0.7205 - loss: 0.5214 - val_accuracy: 0.7310 - val_loss: 0.5093
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 277ms/step - accuracy: 0.7250 - loss: 0.5155 - val_accuracy: 0.7285 - val_loss: 0.5174
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 275ms/step - accuracy: 0.7280 - loss: 0.5128 - val_accuracy: 0.7310 - val_loss: 0.5047
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 272ms/step - accuracy: 0.7290 - loss: 0.5080 - val_accuracy: 0.7335 - val_loss: 0.4985
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 265ms/step - accuracy: 0.7310 - loss: 0.5057 - val_accuracy: 0.7345 - val_loss: 0.4989
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 259ms/step - accuracy: 0.7310 - loss: 0.5028 - 

In [35]:
cnn_process1.save('/content/drive/MyDrive/CornSeeds/corn_seed_quality_detection_v0_0_1.keras')

In [36]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_process1)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('/content/drive/MyDrive/CornSeeds/corn_seed_quality_detection_v0_0_1.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpd_7jd1t5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='keras_tensor_41')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137708585741840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813871632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813871824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813872208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813872400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813872784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813872976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137709813873360: TensorSpec(shape=(), dtype=tf.resource, name=None)


* PROCESSING

In [37]:
import numpy as np
from keras.preprocessing import image

test_images1 = image.load_img('/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/healthy/aug_3_2036712710_WangDataa35.jpg', target_size=(64, 64))
test_images1 = image.img_to_array(test_images1)
test_images1 = np.expand_dims(test_images1, axis=0)
result1 = cnn_process1.predict(test_images1)
training_set1.class_indices
if result1[0][0] == 1:
    prediction = 'healthy'
else:
    prediction = 'unhealthy'
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step
healthy
